# Segmentación Semántica en Videos de Fútbol con U-Net

Entrenamiento de una **U-Net** con encoder **ResNet50** (transfer learning) para segmentar:
- **Jugadores** (diferenciando equipos por color de camiseta)
- **Balón**
- **Portería**

Detecta **goles** mediante reglas geométricas sobre la segmentación.

**Datasets**: Kaggle (football-semantic-segmentation + football-player-segmentation)

**Framework**: TensorFlow/Keras

## 1. Setup y Dependencias

In [ ]:
import os, sys, json, shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    KAGGLE_JSON_SRC = '/content/drive/MyDrive/Kaggle/kaggle.json'
    KAGGLE_JSON_DST = os.path.expanduser('~/.kaggle/kaggle.json')
    os.makedirs(os.path.dirname(KAGGLE_JSON_DST), exist_ok=True)
    shutil.copy(KAGGLE_JSON_SRC, KAGGLE_JSON_DST)
    os.chmod(KAGGLE_JSON_DST, 600)
    print('Kaggle credentials configured.')

    !pip install -q kaggle opencv-python-headless scikit-learn yt-dlp imageio[ffmpeg]

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Descarga de Datasets

In [ ]:
BASE_DIR = '/content/data' if IN_COLAB else './data'
DS1_DIR = os.path.join(BASE_DIR, 'football-semantic-segmentation')
DS2_DIR = os.path.join(BASE_DIR, 'football-player-segmentation')
os.makedirs(DS1_DIR, exist_ok=True)
os.makedirs(DS2_DIR, exist_ok=True)

if IN_COLAB:
    !kaggle datasets download -d sadhliroomyprime/football-semantic-segmentation -p {DS1_DIR} --unzip
    !kaggle datasets download -d ihelon/football-player-segmentation -p {DS2_DIR} --unzip
else:
    os.system(f'kaggle datasets download -d sadhliroomyprime/football-semantic-segmentation -p "{DS1_DIR}" --unzip')
    os.system(f'kaggle datasets download -d ihelon/football-player-segmentation -p "{DS2_DIR}" --unzip')

print('\n=== Dataset 1 ===')
for root, dirs, files in os.walk(DS1_DIR):
    level = root.replace(DS1_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... (+{len(files)-5} more)')

print('\n=== Dataset 2 ===')
for root, dirs, files in os.walk(DS2_DIR):
    level = root.replace(DS2_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... (+{len(files)-5} more)')

## 3. Exploración y Análisis de Datos

In [ ]:
ds1_json = None
for f in os.listdir(DS1_DIR):
    if f.endswith('.json'):
        ds1_json = os.path.join(DS1_DIR, f)
        break

if ds1_json:
    with open(ds1_json, 'r') as f:
        ds1_coco = json.load(f)
    print(f'DS1 JSON: {ds1_json}')
    print(f'  Imágenes: {len(ds1_coco.get("images", []))}')
    print(f'  Anotaciones: {len(ds1_coco.get("annotations", []))}')
    print(f'  Categorías:')
    for cat in ds1_coco.get('categories', []):
        print(f'    {cat["id"]}: {cat["name"]}')
    if ds1_coco.get('annotations'):
        ann = ds1_coco['annotations'][0]
        print(f'\n  Ejemplo anotación:')
        for k, v in ann.items():
            val_str = str(v)[:100]
            print(f'    {k}: {val_str}')
else:
    print('No se encontró JSON en DS1')

ds2_json = os.path.join(DS2_DIR, 'annotations', 'instances_default.json')
if os.path.exists(ds2_json):
    with open(ds2_json, 'r') as f:
        ds2_coco = json.load(f)
    print(f'\nDS2 JSON: {ds2_json}')
    print(f'  Imágenes: {len(ds2_coco.get("images", []))}')
    print(f'  Anotaciones: {len(ds2_coco.get("annotations", []))}')
    print(f'  Categorías:')
    for cat in ds2_coco.get('categories', []):
        print(f'    {cat["id"]}: {cat["name"]}')
else:
    print('No se encontró JSON en DS2')

## 4. Preprocesamiento: Generación de Máscaras desde COCO

Ambos datasets usan formato COCO. Generamos máscaras PNG limpias a partir de los polígonos/RLE.

**Clases objetivo (5 clases):**
| ID | Clase | Descripción |
|----|-------|-------------|
| 0 | background | Fondo |
| 1 | player | Jugadores de campo |
| 2 | goalkeeper | Porteros |
| 3 | ball | Balón |
| 4 | goalpost | Portería |

In [ ]:
IMG_SIZE = (256, 256)
NUM_CLASSES = 5
CLASS_NAMES = ['background', 'player', 'goalkeeper', 'ball', 'goalpost']
CLASS_COLORS = {
    0: [0, 0, 0],
    1: [0, 255, 0],
    2: [255, 255, 0],
    3: [255, 0, 0],
    4: [0, 0, 255],
}

def build_cat_mapping(coco_categories):
    mapping = {}
    for cat in coco_categories:
        name = cat['name'].lower()
        cat_id = cat['id']
        if 'goalkeeper' in name or 'goalie' in name or 'keeper' in name:
            mapping[cat_id] = 2
        elif 'player' in name or 'person' in name or 'human' in name:
            mapping[cat_id] = 1
        elif 'ball' in name:
            mapping[cat_id] = 3
        elif 'goal' in name or 'post' in name or 'net' in name:
            mapping[cat_id] = 4
        elif 'referee' in name:
            mapping[cat_id] = 0
        else:
            mapping[cat_id] = 1
    return mapping

def coco_to_pairs(coco_data, images_dir, masks_dir, ds_name):
    os.makedirs(masks_dir, exist_ok=True)

    cat_mapping = build_cat_mapping(coco_data.get('categories', []))
    print(f'  {ds_name} - Mapeo COCO -> clases:')
    for cat in coco_data.get('categories', []):
        print(f'    {cat["id"]}: {cat["name"]} -> clase {cat_mapping.get(cat["id"], 0)}')

    anns_by_img = {}
    for ann in coco_data.get('annotations', []):
        img_id = ann['image_id']
        anns_by_img.setdefault(img_id, []).append(ann)

    pairs = []
    for img_info in coco_data.get('images', []):
        img_id = img_info['id']
        img_filename = img_info['file_name']
        img_path = os.path.join(images_dir, img_filename)

        if not os.path.exists(img_path):
            candidates = [f for f in os.listdir(images_dir)
                         if os.path.splitext(f)[0] == os.path.splitext(img_filename)[0]]
            if candidates:
                img_path = os.path.join(images_dir, candidates[0])
            else:
                continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        mask = np.zeros((h, w), dtype=np.uint8)
        img_anns = anns_by_img.get(img_id, [])

        for ann in img_anns:
            cat_id = ann.get('category_id', 0)
            class_id = cat_mapping.get(cat_id, 1)

            seg = ann.get('segmentation', None)
            if seg is None:
                continue

            if isinstance(seg, list):
                for polygon in seg:
                    if len(polygon) >= 6:
                        pts = np.array(polygon, dtype=np.int32).reshape(-1, 2)
                        cv2.fillPoly(mask, [pts], class_id)
            elif isinstance(seg, dict):
                from pycocotools import mask as maskUtils
                rle = maskUtils.frPyObjects(seg, h, w)
                binary = maskUtils.decode(rle)
                mask[binary > 0] = class_id

        base_name = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(masks_dir, base_name + '_mask.png')
        cv2.imwrite(mask_path, mask)
        pairs.append((img_path, mask_path))

    return pairs

print('Procesando Dataset 1 (COCO JSON)...')
ds1_masks_dir = os.path.join(DS1_DIR, 'generated_masks')
ds1_images_dir = os.path.join(DS1_DIR, 'images')
pairs_ds1 = coco_to_pairs(ds1_coco, ds1_images_dir, ds1_masks_dir, 'DS1')
print(f'  DS1: {len(pairs_ds1)} pares\n')

print('Procesando Dataset 2 (COCO JSON)...')
ds2_masks_dir = os.path.join(DS2_DIR, 'generated_masks')
ds2_images_dir = os.path.join(DS2_DIR, 'images')
pairs_ds2 = coco_to_pairs(ds2_coco, ds2_images_dir, ds2_masks_dir, 'DS2')
print(f'  DS2: {len(pairs_ds2)} pares\n')

pairs = pairs_ds1 + pairs_ds2
print(f'Total de pares: {len(pairs)}')
if pairs:
    print('\nPrimeros 3 pares:')
    for img, msk in pairs[:3]:
        print(f'  IMG: {os.path.basename(img)}')
        print(f'  MSK: {os.path.basename(msk)}')
        m = cv2.imread(msk, cv2.IMREAD_GRAYSCALE)
        unique, counts = np.unique(m, return_counts=True)
        for u, c in zip(unique, counts):
            print(f'    Clase {u}: {c} px ({c/(m.shape[0]*m.shape[1])*100:.1f}%)')
        print()

In [ ]:
def show_pairs(pairs, n=4, title='Muestras'):
    n = min(n, len(pairs))
    if n == 0:
        print('No hay pares para mostrar')
        return
    fig, axes = plt.subplots(n, 2, figsize=(12, 5*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        img_path, mask_path = pairs[i]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'Imagen: {os.path.basename(img_path)}')
        axes[i, 0].axis('off')

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        colored = np.zeros((*mask.shape, 3), dtype=np.uint8)
        for cls_id, color in CLASS_COLORS.items():
            colored[mask == cls_id] = color
        axes[i, 1].imshow(colored)
        axes[i, 1].set_title(f'Máscara: {os.path.basename(mask_path)}')
        axes[i, 1].axis('off')

    patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
    plt.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5)
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

if pairs_ds1:
    show_pairs(pairs_ds1[:4], title='Dataset 1: Football Semantic Segmentation')
if pairs_ds2:
    show_pairs(pairs_ds2[:4], title='Dataset 2: Football Player Segmentation')

In [ ]:
def analyze_class_distribution(pairs, n=50):
    total_pixels = np.zeros(NUM_CLASSES)
    imgs_with_class = np.zeros(NUM_CLASSES)
    for img_path, mask_path in pairs[:n]:
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        for c in range(NUM_CLASSES):
            count = np.sum(mask == c)
            total_pixels[c] += count
            if count > 0:
                imgs_with_class[c] += 1

    total = total_pixels.sum()
    print('Distribución de clases (primeras {} imágenes):'.format(min(n, len(pairs))))
    for c in range(NUM_CLASSES):
        pct = total_pixels[c] / total * 100 if total > 0 else 0
        print(f'  {CLASS_NAMES[c]:12s}: {total_pixels[c]:>10.0f} px ({pct:5.1f}%) | presente en {imgs_with_class[c]:.0f} imgs')
    return total_pixels

class_dist = analyze_class_distribution(pairs)

## 5. Preparación del Dataset

In [ ]:
def load_image(img_path, target_size=IMG_SIZE):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    return img.astype(np.float32) / 255.0

def load_mask(mask_path, target_size=IMG_SIZE):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return np.zeros((*target_size, NUM_CLASSES), dtype=np.float32)
    mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    mask = np.clip(mask, 0, NUM_CLASSES - 1)
    return tf.keras.utils.to_categorical(mask, NUM_CLASSES).astype(np.float32)

def augment(image, mask):
    if np.random.random() > 0.5:
        image = np.fliplr(image).copy()
        mask = np.fliplr(mask).copy()

    if np.random.random() > 0.5:
        brightness = np.random.uniform(0.7, 1.3)
        image = np.clip(image * brightness, 0, 1).astype(np.float32)

    if np.random.random() > 0.5:
        contrast = np.random.uniform(0.7, 1.3)
        mean_val = np.mean(image, axis=(0, 1), keepdims=True)
        image = np.clip((image - mean_val) * contrast + mean_val, 0, 1).astype(np.float32)

    if np.random.random() > 0.6:
        angle = np.random.uniform(-15, 15)
        h, w = image.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        image = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        mask = cv2.warpAffine(mask, M, (w, h), borderMode=cv2.BORDER_CONSTANT)
        if len(mask.shape) == 2:
            mask = np.expand_dims(mask, -1)

    if np.random.random() > 0.7:
        scale = np.random.uniform(0.8, 1.2)
        h, w = image.shape[:2]
        new_h, new_w = int(h * scale), int(w * scale)
        img_scaled = cv2.resize(image, (new_w, new_h))
        msk_scaled = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        if len(msk_scaled.shape) == 2:
            msk_scaled = np.expand_dims(msk_scaled, -1)

        result_img = np.zeros_like(image)
        result_msk = np.zeros_like(mask)

        src_y = max(0, (new_h - h) // 2)
        src_x = max(0, (new_w - w) // 2)
        dst_y = max(0, (h - new_h) // 2)
        dst_x = max(0, (w - new_w) // 2)
        copy_h = min(h, new_h)
        copy_w = min(w, new_w)

        result_img[dst_y:dst_y+copy_h, dst_x:dst_x+copy_w] = img_scaled[src_y:src_y+copy_h, src_x:src_x+copy_w]
        result_msk[dst_y:dst_y+copy_h, dst_x:dst_x+copy_w] = msk_scaled[src_y:src_y+copy_h, src_x:src_x+copy_w]
        image = result_img
        mask = result_msk

    return image.astype(np.float32), mask.astype(np.float32)

def prepare_dataset(pairs_list, augment_data=False, oversample=3):
    images, masks = [], []
    for img_path, mask_path in pairs_list:
        img = load_image(img_path)
        msk = load_mask(mask_path)

        has_minority = False
        for c in [2, 3, 4]:
            if np.sum(msk[:, :, c]) > 50:
                has_minority = True
                break

        n_copies = oversample if (has_minority and augment_data) else 1

        for _ in range(n_copies):
            if augment_data:
                img_aug, msk_aug = augment(img.copy(), msk.copy())
            else:
                img_aug, msk_aug = img, msk
            images.append(img_aug)
            masks.append(msk_aug)

    return np.array(images), np.array(masks)

train_pairs, temp_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42)

print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

X_train, y_train = prepare_dataset(train_pairs, augment_data=True, oversample=3)
X_val, y_val = prepare_dataset(val_pairs, augment_data=False)
X_test, y_test = prepare_dataset(test_pairs, augment_data=False)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val: {X_val.shape} | y_val: {y_val.shape}')
print(f'X_test: {X_test.shape} | y_test: {y_test.shape}')

In [ ]:
BATCH_SIZE = 8

def make_tf_dataset(X, y, batch_size=BATCH_SIZE, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(X_train, y_train)
val_ds = make_tf_dataset(X_val, y_val, shuffle=False)
test_ds = make_tf_dataset(X_test, y_test, shuffle=False)

for bx, by in train_ds.take(1):
    print(f'Batch: images={bx.shape}, masks={by.shape}')

## 6. Construcción de la U-Net con ResNet50

In [ ]:
def compute_class_weights(y_train_data):
    class_pixels = np.zeros(NUM_CLASSES)
    for i in range(len(y_train_data)):
        mask = y_train_data[i]
        cls = np.argmax(mask, axis=-1)
        for c in range(NUM_CLASSES):
            class_pixels[c] += np.sum(cls == c)

    total = class_pixels.sum()
    weights = np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        if class_pixels[c] > 0:
            weights[c] = total / (NUM_CLASSES * class_pixels[c])
        else:
            weights[c] = 1.0

    weights = np.clip(weights, 1.0, 50.0)
    print('Class weights:')
    for c in range(NUM_CLASSES):
        print(f'  {CLASS_NAMES[c]:12s}: {weights[c]:.2f} ({class_pixels[c]:.0f} px)')
    return weights

class_weights = compute_class_weights(y_train)

In [ ]:
def weighted_dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_flat = tf.reshape(y_true, [-1, NUM_CLASSES])
    y_pred_flat = tf.reshape(y_pred, [-1, NUM_CLASSES])

    w = tf.constant(class_weights, dtype=tf.float32)

    total_loss = 0.0
    for i in range(NUM_CLASSES):
        true_c = y_true_flat[:, i]
        pred_c = y_pred_flat[:, i]
        intersection = tf.reduce_sum(true_c * pred_c)
        dice = (2.0 * intersection + smooth) / (tf.reduce_sum(true_c) + tf.reduce_sum(pred_c) + smooth)
        total_loss += w[i] * (1.0 - dice)

    return total_loss / tf.reduce_sum(w)

def focal_loss(y_true, y_pred, gamma=2.0):
    y_true_flat = tf.reshape(y_true, [-1, NUM_CLASSES])
    y_pred_flat = tf.reshape(y_pred, [-1, NUM_CLASSES])

    epsilon = 1e-7
    y_pred_flat = tf.clip_by_value(y_pred_flat, epsilon, 1.0 - epsilon)

    ce = -y_true_flat * tf.math.log(y_pred_flat)
    weight = tf.pow(1.0 - y_pred_flat, gamma)
    fl = weight * ce

    w = tf.constant(class_weights, dtype=tf.float32)
    fl = fl * w

    return tf.reduce_mean(fl)

def combined_loss(y_true, y_pred):
    return weighted_dice_loss(y_true, y_pred) + focal_loss(y_true, y_pred)

def mean_iou_metric(y_true, y_pred):
    y_pred_class = tf.argmax(y_pred, axis=-1)
    y_true_class = tf.argmax(y_true, axis=-1)
    iou_values = []
    for cls in range(NUM_CLASSES):
        pred_mask = tf.cast(tf.equal(y_pred_class, cls), tf.float32)
        true_mask = tf.cast(tf.equal(y_true_class, cls), tf.float32)
        intersection = tf.reduce_sum(pred_mask * true_mask)
        union = tf.reduce_sum(pred_mask) + tf.reduce_sum(true_mask) - intersection
        iou = (intersection + 1e-6) / (union + 1e-6)
        iou_values.append(iou)
    return tf.reduce_mean(iou_values)

In [ ]:
def build_unet(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = tf.keras.layers.Input(shape=input_shape)

    resnet = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )

    skip_names = ['conv1_relu', 'conv2_block3_out', 'conv3_block4_out', 'conv4_block6_out']
    skip_layers = [resnet.get_layer(n).output for n in skip_names]
    bridge = resnet.get_layer('conv5_block3_out').output

    def decoder_block(x, skip, filters):
        x = tf.keras.layers.Conv2DTranspose(filters, (3, 3), strides=(2, 2), padding='same')(x)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        return x

    x = bridge
    x = decoder_block(x, skip_layers[3], 512)
    x = decoder_block(x, skip_layers[2], 256)
    x = decoder_block(x, skip_layers[1], 128)
    x = decoder_block(x, skip_layers[0], 64)
    x = tf.keras.layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same')(x)
    outputs = tf.keras.layers.Conv2D(num_classes, (1, 1), activation='softmax')(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = build_unet()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[mean_iou_metric, 'accuracy']
)
model.summary()

## 7. Entrenamiento

In [ ]:
CHECKPOINT_DIR = '/content/drive/MyDrive/Unet/checkpoints' if IN_COLAB else './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_unet.keras')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1),
]

EPOCHS = 50

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['mean_iou_metric'], label='Train mIoU')
axes[1].plot(history.history['val_mean_iou_metric'], label='Val mIoU')
axes[1].set_title('Mean IoU')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history.history['accuracy'], label='Train Accuracy')
axes[2].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[2].set_title('Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 8. Evaluación del Modelo

In [ ]:
test_loss, test_iou, test_acc = model.evaluate(test_ds, verbose=1)
print(f'\n=== Resultados en Test Set ===')
print(f'Loss: {test_loss:.4f}')
print(f'Mean IoU: {test_iou:.4f}')
print(f'Accuracy: {test_acc:.4f}')

In [ ]:
def compute_per_class_iou(y_true, y_pred):
    y_true_class = np.argmax(y_true, axis=-1)
    y_pred_class = np.argmax(y_pred, axis=-1)
    ious = {}
    for cls in range(NUM_CLASSES):
        pred_mask = (y_pred_class == cls).astype(np.float32)
        true_mask = (y_true_class == cls).astype(np.float32)
        intersection = np.sum(pred_mask * true_mask)
        union = np.sum(pred_mask) + np.sum(true_mask) - intersection
        ious[CLASS_NAMES[cls]] = (intersection + 1e-6) / (union + 1e-6)
    return ious

y_pred_test = model.predict(test_ds, verbose=1)
class_ious = compute_per_class_iou(y_test, y_pred_test)

print('\n=== IoU por Clase ===')
for cls_name, iou in class_ious.items():
    print(f'{cls_name}: {iou:.4f}')

In [ ]:
def visualize_predictions(images, true_masks, pred_masks, n=4):
    n = min(n, len(images))
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        img = (images[i] * 255).astype(np.uint8)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Imagen Original')
        axes[i, 0].axis('off')

        true_class = np.argmax(true_masks[i], axis=-1)
        colored_true = np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
        for cls_id, color in CLASS_COLORS.items():
            colored_true[true_class == cls_id] = color
        axes[i, 1].imshow(colored_true)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')

        pred_class = np.argmax(pred_masks[i], axis=-1)
        colored_pred = np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
        for cls_id, color in CLASS_COLORS.items():
            colored_pred[pred_class == cls_id] = color
        axes[i, 2].imshow(colored_pred)
        axes[i, 2].set_title('Predicción')
        axes[i, 2].axis('off')

    legend_patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
    plt.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=5)
    plt.tight_layout()
    plt.show()

n_show = min(4, len(X_test))
visualize_predictions(X_test[:n_show], y_test[:n_show], y_pred_test[:n_show], n=n_show)

## 9. Post-procesamiento: Diferenciación de Equipos

Se usa **K-means (k=2)** en espacio **HSV** sobre los píxeles de camiseta de cada jugador segmentado.

In [ ]:
TEAM_COLORS = {
    0: [0, 0, 0],
    1: [0, 255, 0],
    2: [255, 255, 0],
    3: [255, 0, 0],
    4: [0, 0, 255],
    5: [0, 255, 255],
    6: [255, 0, 255],
}
TEAM_NAMES = ['background', 'player', 'goalkeeper', 'ball', 'goalpost', 'Team A', 'Team B']

def differentiate_teams(image, pred_mask):
    pred_class = np.argmax(pred_mask, axis=-1)
    team_mask = pred_class.copy()

    player_pixels = image[pred_class == 1]
    if len(player_pixels) < 20:
        return team_mask

    player_pixels_uint8 = (player_pixels * 255).astype(np.uint8)
    hsv_pixels = cv2.cvtColor(
        player_pixels_uint8.reshape(1, -1, 3), cv2.COLOR_RGB2HSV
    ).reshape(-1, 3)

    valid = (hsv_pixels[:, 1] > 30) & (hsv_pixels[:, 2] > 30)
    filtered = hsv_pixels[valid]

    if len(filtered) < 20:
        return team_mask

    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    kmeans.fit(filtered)

    player_hsv = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    player_indices = np.where(pred_class == 1)

    for idx in range(len(player_indices[0])):
        y, x = player_indices[0][idx], player_indices[1][idx]
        pixel_hsv = player_hsv[y, x].reshape(1, -1)
        if pixel_hsv[0, 1] > 30 and pixel_hsv[0, 2] > 30:
            team_label = kmeans.predict(pixel_hsv)[0]
            team_mask[y, x] = 5 + team_label

    return team_mask

if len(X_test) > 0:
    sample_img = X_test[0]
    sample_pred = y_pred_test[0]
    team_result = differentiate_teams(sample_img, sample_pred)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow((sample_img * 255).astype(np.uint8))
    axes[0].set_title('Imagen Original')
    axes[0].axis('off')

    colored = np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
    for cls_id, color in TEAM_COLORS.items():
        colored[team_result == cls_id] = color
    axes[1].imshow(colored)
    axes[1].set_title('Segmentación por Equipos')
    axes[1].axis('off')

    patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in zip(TEAM_NAMES, TEAM_COLORS.values())]
    plt.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=4)
    plt.tight_layout()
    plt.show()

## 10. Detección de Goles con Reglas Geométricas

Si el centroide del balón cruza la línea de gol dentro de la portería segmentada → **GOL**

In [ ]:
class GoalDetector:
    def __init__(self):
        self.ball_history = []
        self.goal_events = []
        self.goalpost_history = []

    def get_ball_centroid(self, pred_mask):
        pred_class = np.argmax(pred_mask, axis=-1)
        ball_pixels = np.where(pred_class == 3)
        if len(ball_pixels[0]) == 0:
            return None
        return (int(np.mean(ball_pixels[1])), int(np.mean(ball_pixels[0])))

    def get_goalpost_region(self, pred_mask):
        pred_class = np.argmax(pred_mask, axis=-1)
        goal_pixels = np.where(pred_class == 4)
        if len(goal_pixels[0]) == 0:
            return None
        return (int(np.min(goal_pixels[1])), int(np.min(goal_pixels[0])),
                int(np.max(goal_pixels[1])), int(np.max(goal_pixels[0])))

    def detect_goal(self, pred_mask, frame_idx, timestamp=None):
        ball_pos = self.get_ball_centroid(pred_mask)
        goal_region = self.get_goalpost_region(pred_mask)

        if ball_pos is None:
            self.ball_history.append(None)
            return False

        self.ball_history.append(ball_pos)

        if goal_region is None or len(self.ball_history) < 3:
            self.goalpost_history.append(goal_region)
            return False

        gx_min, gy_min, gx_max, gy_max = goal_region
        bx, by = ball_pos

        prev_positions = [p for p in self.ball_history[-5:-1] if p is not None]
        if len(prev_positions) < 2:
            self.goalpost_history.append(goal_region)
            return False

        prev_x = np.mean([p[0] for p in prev_positions])
        inside_vertical = gy_min <= by <= gy_max
        inside_horizontal = gx_min <= bx <= gx_max

        goal_scored = False
        crossed_left = (prev_x < gx_min and bx >= gx_min)
        crossed_right = (prev_x > gx_max and bx <= gx_max)

        if (crossed_left or crossed_right) and inside_vertical:
            goal_scored = True
        elif inside_horizontal and inside_vertical:
            if self.goalpost_history and self.goalpost_history[-1] is not None:
                pgx_min, _, pgx_max, _ = self.goalpost_history[-1]
                if not (pgx_min <= prev_x <= pgx_max):
                    goal_scored = True

        self.goalpost_history.append(goal_region)

        if goal_scored:
            if not self.goal_events or (frame_idx - self.goal_events[-1]['frame']) > 15:
                self.goal_events.append({
                    'frame': frame_idx,
                    'timestamp': timestamp,
                    'ball_position': ball_pos,
                    'goalpost_region': goal_region
                })
                return True
        return False

    def reset(self):
        self.ball_history = []
        self.goal_events = []
        self.goalpost_history = []

goal_detector = GoalDetector()
print('GoalDetector inicializado.')

## 11. Descarga y Procesamiento de Video

Se descarga automáticamente un video de highlights de fútbol y se procesa frame a frame.
El video resultado se **guarda como MP4** en Google Drive.

In [ ]:
VIDEO_DIR = '/content/videos' if IN_COLAB else './videos'
OUTPUT_DIR = '/content/drive/MyDrive/Unet/output' if IN_COLAB else './output'
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

VIDEO_URLS = [
    'https://www.youtube.com/watch?v=Dh3UQv9yJKE',
    'https://www.youtube.com/watch?v=74RskmMGQ9E',
    'https://www.youtube.com/watch?v=QkGXYnIiYjE',
]

VIDEO_PATH = os.path.join(VIDEO_DIR, 'match.mp4')

if not os.path.exists(VIDEO_PATH):
    print('Descargando video de highlights de fútbol...')
    downloaded = False
    for url in VIDEO_URLS:
        try:
            print(f'\nIntentando: {url}')
            !yt-dlp -f "mp4[height<=480]/best[height<=480]" --merge-output-format mp4 -o "{VIDEO_PATH}" "{url}"
            if os.path.exists(VIDEO_PATH) and os.path.getsize(VIDEO_PATH) > 1000000:
                downloaded = True
                print(f'Video descargado: {VIDEO_PATH} ({os.path.getsize(VIDEO_PATH)/1024/1024:.1f} MB)')
                break
            elif os.path.exists(VIDEO_PATH):
                os.remove(VIDEO_PATH)
        except Exception as e:
            print(f'Error: {e}')
            if os.path.exists(VIDEO_PATH):
                os.remove(VIDEO_PATH)

    if not downloaded:
        print('\nNo se pudo descargar automáticamente. Intenta con otro URL:')
        print('  !yt-dlp -f "mp4[height<=480]" --merge-output-format mp4 -o "{VIDEO_PATH}" "NUEVA_URL"')
        print('\nO sube un video manualmente:')
        if IN_COLAB:
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                shutil.move(fname, VIDEO_PATH)
                break
else:
    print(f'Video ya existe: {VIDEO_PATH}')

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f'\nVideo: {VIDEO_PATH}')
    print(f'FPS: {fps} | Frames: {total_frames} | Resolución: {width}x{height}')
    print(f'Duración: {total_frames/fps:.1f} segundos')

In [ ]:
def process_video(video_path, model, output_path, goal_det, img_size=IMG_SIZE):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    goal_det.reset()
    frame_idx = 0
    goals_detected = 0
    goal_flash = 0

    print(f'Procesando {total_frames} frames...')

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, img_size)
        frame_norm = frame_resized.astype(np.float32) / 255.0
        pred = model.predict(np.expand_dims(frame_norm, 0), verbose=0)[0]

        team_mask = differentiate_teams(frame_norm, pred)

        overlay = np.zeros((*img_size, 3), dtype=np.uint8)
        for cls_id, color in TEAM_COLORS.items():
            overlay[team_mask == cls_id] = color

        overlay_full = cv2.resize(overlay, (width, height), interpolation=cv2.INTER_NEAREST)
        blended = cv2.addWeighted(frame, 0.6, overlay_full, 0.4, 0)

        timestamp = frame_idx / fps
        if goal_det.detect_goal(pred, frame_idx, timestamp):
            goals_detected += 1
            goal_flash = int(fps * 2)
            print(f'  GOL en frame {frame_idx} (t={timestamp:.1f}s)')

        if goal_flash > 0:
            cv2.putText(blended, 'GOL!!!', (width//2 - 120, height//2),
                       cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 0, 255), 5)
            goal_flash -= 1

        ball_pos = goal_det.get_ball_centroid(pred)
        if ball_pos is not None:
            bx = int(ball_pos[0] * width / img_size[1])
            by = int(ball_pos[1] * height / img_size[0])
            cv2.circle(blended, (bx, by), 8, (255, 0, 0), -1)

        goal_region = goal_det.get_goalpost_region(pred)
        if goal_region is not None:
            gx1 = int(goal_region[0] * width / img_size[1])
            gy1 = int(goal_region[1] * height / img_size[0])
            gx2 = int(goal_region[2] * width / img_size[1])
            gy2 = int(goal_region[3] * height / img_size[0])
            cv2.rectangle(blended, (gx1, gy1), (gx2, gy2), (0, 0, 255), 2)

        info = f'Frame: {frame_idx}/{total_frames} | Goles: {goals_detected} | t={timestamp:.1f}s'
        cv2.putText(blended, info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(blended)
        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f'  {frame_idx}/{total_frames} ({frame_idx/total_frames*100:.1f}%)')

    cap.release()
    out.release()

    print(f'\n=== Resumen ===')
    print(f'Frames procesados: {frame_idx}')
    print(f'Goles detectados: {goals_detected}')
    print(f'Video guardado en: {output_path}')
    return goals_detected, goal_det.goal_events

OUTPUT_VIDEO_PATH = os.path.join(OUTPUT_DIR, 'resultado_segmentacion_goles.mp4')

if os.path.exists(VIDEO_PATH):
    goals, events = process_video(VIDEO_PATH, model, OUTPUT_VIDEO_PATH, goal_detector)
else:
    print('No hay video para procesar.')

In [ ]:
if goal_detector.goal_events:
    print('=== Goles Detectados ===')
    for i, event in enumerate(goal_detector.goal_events, 1):
        print(f'Gol {i}: Frame {event["frame"]} | t={event["timestamp"]:.1f}s | balón={event["ball_position"]}')
else:
    print('No se detectaron goles en el video.')

print(f'\nVideo guardado en: {OUTPUT_VIDEO_PATH}')

if IN_COLAB and os.path.exists(OUTPUT_VIDEO_PATH):
    print(f'Tamaño: {os.path.getsize(OUTPUT_VIDEO_PATH) / (1024*1024):.1f} MB')
    print('Disponible en Google Drive: Unet/output/resultado_segmentacion_goles.mp4')

## 12. Resumen y Conclusiones

In [ ]:
print('=' * 60)
print('RESUMEN DE RESULTADOS')
print('=' * 60)

print(f'\n--- Modelo ---')
print(f'Arquitectura: U-Net + ResNet50 (ImageNet)')
print(f'Clases: {NUM_CLASSES} ({CLASS_NAMES})')
print(f'Input: {IMG_SIZE}')
print(f'Epochs: {len(history.history["loss"])}')

print(f'\n--- Métricas en Test ---')
print(f'Loss: {test_loss:.4f}')
print(f'Mean IoU: {test_iou:.4f}')
print(f'Accuracy: {test_acc:.4f}')

print(f'\n--- IoU por Clase ---')
for cls_name, iou in class_ious.items():
    print(f'  {cls_name}: {iou:.4f}')

print(f'\n--- Detección de Goles ---')
print(f'Goles detectados: {len(goal_detector.goal_events) if goal_detector.goal_events else 0}')

print(f'\n--- Archivos ---')
print(f'Modelo: {checkpoint_path}')
print(f'Video: {OUTPUT_VIDEO_PATH}')

print(f'\n--- Fortalezas ---')
print('- Transfer learning con ResNet50')
print('- Focal Loss + Weighted Dice para clases desbalanceadas')
print('- Oversampling de imágenes con clases minoritarias')
print('- K-means HSV para diferenciar equipos')

print(f'\n--- Mejoras Posibles ---')
print('- Más datos de entrenamiento')
print('- Data augmentation más agresiva (mosaic, cutmix)')
print('- Tracking temporal (LSTM/GRU) para goles')
print('- Encoder más profundo (EfficientNet-B4)')

print('\n' + '=' * 60)